In [ ]:
#Dzisiaj będe robić mój pierwszy model QSAR wygenerwoany przez Gemini
#Uczenie Maszynowe (ML-Machine Learning) jest to narzedzie sztucznej inteligencji w ktorym uczymy komputer-
#- prace z danymi zamiast otrzymywać sztywne instrukcję
#W chemii oznacza to analizę tysięcy znanych reakcji i cząsteczek, aby komputer sam 
#przewidywał nowe właściwości lub wyniki bez konieczności kosztownych testów w laboratorium (uproszczenie)

In [17]:
#Te narzędzia juz znam
from rdkit import Chem
import pandas as pd
from rdkit.Chem import Descriptors
from rdkit.Chem import rdFingerprintGenerator
#Nowe Narzędzia do AI
from sklearn.model_selection import train_test_split#Dobieramy funkcję, która automatycznie potnie nasz zbiór danych na dwie części: "Materiały do nauki" (treningowe) i "Sprawdzian" (testowe).
from sklearn.ensemble import RandomForestRegressor#Importujemy sam algorytm sztucznej inteligencji. Random Forest (Las Losowy) to zbiór setek "drzew decyzyjnych", które wspólnie głosują nad ostatecznym wynikiem. Słowo Regressor oznacza, że będziemy przewidywać konkretną liczbę (LogP), a nie kategorię (np. "toksyczny" / "nietoksyczny").
#Robie listę naszych cząsteczek
smiles_list = ["C",
               "CC",
               "CCC",
               "CCCC",
               "CCCCC",
               "c1ccccc1",
               "CCO",
               "CCCO",
               "CCC(=O)",
               "CCN"
              ]
#Przygotowujęmy fingerprint promien otocznia=2, długość bitów=1024
morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)
#Tworzę puste listy do których będe wrzucać gotowe dane
X_dane_wejściowe = []
Y_cel_do_przewidzenia = []
#Pętla do przejścia przez każdą cząsteczke
for sml in smiles_list:
    mol= Chem.MolFromSmiles(sml) #Tworzę obraz molekul z smiles_list

    fp = morgan_gen.GetFingerprint(mol) #Generuje Fingerprinty
    fp_lista = list(fp) #Zmieniam specyficzny rdkitowy format na zwykła pythonową liste zer i jedynek
    X_dane_wejściowe.append(fp_lista) #Wrzucam do naszej pustej listy
    prawdziwy_logp = Descriptors.MolLogP(mol) #Obliczam logp dla moich czasteczek
    Y_cel_do_przewidzenia.append(prawdziwy_logp) #Wrzucamy wynik do naszej listy Y
    
#Zanim zaczniemy uczyć model, musimy ukryć przed nim część danych. Inaczej po prostu
#"nauczyłby się ich na pamięć" (to tzw. overfitting),a my nie wiedzielibyśmy, 
#czy potrafi logicznie myśleć o nowych cząsteczkach
#train.test_split służy do podziału zbioru danych na dwie części: część treningową i część testową.
X_trening, X_test, Y_trening, Y_test, smiles_trening, smiles_test = train_test_split( #zwraca mi 4 nowe zmienne (po lewej stronie =) X_trening i Y_trening to struktury ktore model wykorzysta aby sie nauczyć, zaś X_test i Y_test to struktury ukryte model bedzie przewidywac
    X_dane_wejściowe,                  
    Y_cel_do_przewidzenia,
    smiles_list,
    test_size=0.2, #Określa, jaki procent danych ma trafić do zbioru testowego. W tym przypadku 0.2 oznacza 20% na test i 80% na trening. (Możesz też użyć train_size).
    random_state=42 #Działa jak "ziarno" (seed) dla generatora liczb losowych. Sprawia, że za każdym razem, gdy uruchomisz ten kod, podział będzie dokładnie taki sam. To kluczowe, jeśli chcesz, aby Twoje wyniki były powtarzalne.
    )
#Tutaj juz zaczyna sie machine learning - uczę model
model = RandomForestRegressor(random_state=42) #Powołujemy do życia nasz algorytm. Na razie to pusta "kartka", nic nie wie o chemii.
model.fit(X_trening, Y_trening) #Najważniejsza funkcja w uczeniu maszynowym. fit (z ang. dopasuj). Podajemy mu kody kreskowe (X_trening) i prawdziwe wyniki (Y_trening). Algorytm analizuje te dane, szuka w nich prawidłowości i wewnętrznie dostraja swoje rówania matematyczne. Trwa proces nauki.
przewidywanie = model.predict(X_test) #Druga najważniejsza funkcja. predict (przewiduj). Bierzemy wyuczony model i dajemy mu X_test (same kody kreskowe cząsteczek, których model nigdy wcześniej nie widział). Model oddaje nam listę zgadniętych wartości LogP.

tabela_wynikow = pd.DataFrame({ #Zestawiamy w ładnej tabeli Pandas wyniki prawdziwe (te, które na początku wyliczyliśmy i ukryliśmy) z wynikami przewidzianymi przez model, aby móc je naocznie porównać.
    "Cząsteczka(smiles)": smiles_test,
    "Prawdziwe LogP (Y_test)": Y_test,
    "Zgadniete przez AI": przewidywanie
    })
display(tabela_wynikow) # wyswietla tabele 
#mozna dodac jeszcze tutaj MAE (Mean Absolute Error) po polsku średni błąd bezwzględny



,Cząsteczka(smiles),Prawdziwe LogP (Y_test),Zgadniete przez AI
0,CCC(=O),0.5953,1.005909
1,CC,1.0262,1.075888


In [ ]:
#Korzystajac z Gemini utworzyłem swój pierwszy model QSAR 
#1) Przekształciłem Smiles w postać macierzową. Utworzyłem zmienne objaśniające X (1024bitowe wektory Morgan Fingerprints) oraz Y (Rzeczywiste wartości LogP)
#2) Podzieliłem zbiory (train_test_split) 80% danych zadałem jako zbiór treningowy do optymalizacji modelu oraz 20% jako zbiór testowy, który służył wyłącznie do końcowej walidacji
#3) Za pomocą funkcji .fit zainiciowałem algorytm RandomForestReggresor i przekazałem mu zbiór treningowy. Model tutaj sam odnalazł nieliniowe korelacje miedzy strukturą wektora X a wartością Y
#4) Wytrenowany model zastosowałem na zbiorze testowym, generując przewidywane wartości LogP. Zestawiłem je z danymi rzeczywistymi aby sprawdzić zdolność modelu do generalizacji
#Dosyc jest to fascynujące ze tak z małej bazy 10 zwiazków maszyna odgadła ten logp w pierwszym jest pomyłka ~0.41 ale drugi ~0,05, imponujące to jest.
#Widze już jak bede mógł wykorzystać Machine Learning w mojej pracy magisterskiej i jak porownac do siebie te wyniki. Bedzie to ciekawy projekt :D